# LangGraph Module · Day 07 — Streaming

**Phase 1 · Module 1: LangGraph & LangChain Deep Dive** · ~2.5 hrs

### Today's tasks (from the plan)
- [ ] Enable **token-level streaming** from a LangGraph agent
- [ ] Use **`.astream_events()`** to capture *every* event the graph emits
- [ ] Build a simple **terminal UI** that streams the agent's thoughts as they happen

**Python skill that feeds this (Day 07):** generators & async generators — `graph.astream(...)` and `graph.astream_events(...)` are async generators you drive with `async for`.

## 0. Setup

Streaming is a **runtime feature of the graph**, not of any one model — so most of this notebook runs with **no API key** using a `GenericFakeChatModel` that streams token-by-token exactly like a real one. The final section swaps in real Gemini streaming, guarded by `HAS_KEY`.

In [1]:
from dotenv import load_dotenv
import os

load_dotenv()
HAS_KEY = bool(os.getenv('GOOGLE_API_KEY'))
print('langgraph streaming notebook')
print('GOOGLE_API_KEY:', 'loaded \u2705' if HAS_KEY else 'MISSING \u274c (only the last section needs it)')

langgraph streaming notebook
GOOGLE_API_KEY: loaded ✅


## 1. The one big idea — *watch the graph work, don't wait for the end*

Everything so far called the graph and read the **final state** after it finished. Fine for a script; bad for a user. A plan-execute run (Day 06) can take many seconds — the user should see the plan, then each step, then tokens of the answer **as they happen**, not a frozen screen followed by a wall of text.

LangGraph exposes the run as a **stream**: `graph.stream(...)` (sync) and `graph.astream(...)` (async) are **generators** — the exact Python pattern from today's 30-min skill. You `for`/`async for` over them and each yield is a slice of progress.

## 2. Three questions → three `stream_mode`s

The **same** stream answers different questions depending on `stream_mode`:

| `stream_mode` | Each yield is… | Use it to… |
|---------------|----------------|-------------|
| `'values'` | the **whole state** after each step | snapshot progress / debug state |
| `'updates'` | only **what each node changed** | see node-by-node output as it lands |
| `'messages'` | **(token, metadata)** pairs from any LLM in a node | stream the answer **token by token** |

We'll build one small agent and view it through all three.

## 3. A tiny agent to stream — fraud-alert triage

Two nodes, a realistic banking job:
1. **`triage`** — classify an incoming transaction alert (rule-based, instant).
2. **`respond`** — an LLM writes the customer-facing message (this is what streams token by token).

State carries the running `messages` list (Day 02's `add_messages` reducer) plus the triage `risk` verdict.

In [2]:
from typing import TypedDict, Annotated
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langchain_core.messages import AIMessage, HumanMessage
from langchain_core.language_models.fake_chat_models import GenericFakeChatModel

class TriageState(TypedDict):
    messages: Annotated[list, add_messages]   # conversation, auto-appended
    risk: str                                 # triage verdict

def _reply_model(text: str):
    # A fake chat model that STREAMS `text` token-by-token - no API key.
    # Swap for ChatGoogleGenerativeAI and the graph is unchanged (see \u00a78).
    return GenericFakeChatModel(messages=iter([AIMessage(content=text)]))

print('state + fake streaming model ready')

state + fake streaming model ready


In [3]:
def triage(state: TriageState) -> dict:
    alert = state['messages'][-1].content.lower()
    risk = 'HIGH' if ('overseas' in alert or 'high-value' in alert) else 'LOW'
    return {'risk': risk}                     # only the field this node changes

def respond(state: TriageState) -> dict:
    risk = state['risk']
    draft = (f'We spotted a {risk}-risk transaction on your account. '
             'We have paused it for review and sent a confirmation code to your phone.')
    model = _reply_model(draft)
    reply = model.invoke(state['messages'])   # the LLM call whose tokens we stream
    return {'messages': [reply]}

builder = StateGraph(TriageState)
builder.add_node('triage', triage)
builder.add_node('respond', respond)
builder.add_edge(START, 'triage')
builder.add_edge('triage', 'respond')
builder.add_edge('respond', END)
graph = builder.compile()
print(graph.get_graph().draw_mermaid())

---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	triage(triage)
	respond(respond)
	__end__([<p>__end__</p>]):::last
	__start__ --> triage;
	triage --> respond;
	respond --> __end__;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc



That wiring renders as a straight line — `triage` then `respond` — which is all we need to watch streaming behave:

```mermaid
flowchart TD
    START([__start__]) --> triage
    triage --> respond
    respond --> END([__end__])
```

☝ Note `triage` returns only `{'risk': ...}` and `respond` returns only `{'messages': [...]}`. Each node updates **just its slice** of state — that's exactly what `stream_mode='updates'` will show us next.

## 4. `stream_mode='updates'` — see each node land

The simplest useful stream: one yield **per node**, containing only what that node changed. You watch the graph advance node by node instead of waiting for the end.

In [4]:
alert = {'messages': [HumanMessage(content='Overseas high-value purchase flagged')],
         'risk': ''}

for update in graph.stream(alert, stream_mode='updates'):
    node = list(update)[0]                     # {'triage': {...}} -> 'triage'
    print(f'[{node:8}] ->', update[node])

[triage  ] -> {'risk': 'HIGH'}
[respond ] -> {'messages': [AIMessage(content='We spotted a HIGH-risk transaction on your account. We have paused it for review and sent a confirmation code to your phone.', additional_kwargs={}, response_metadata={}, id='lc_run--019f386d-d2ea-74e3-9f26-501343531471-0', tool_calls=[], invalid_tool_calls=[])]}


☝ Two yields, one per node, **in execution order**. `triage` lands its `risk` verdict first; then `respond` lands the full message. This is the node-level heartbeat of the run — great for a progress log (“triage done… drafting reply…”).

## 5. `stream_mode='messages'` — token by token

To stream the *answer itself*, switch to `'messages'`. Now each yield is a **`(chunk, metadata)`** pair: `chunk` is one token of an LLM's output, `metadata` tells you which node produced it. This is the mode that powers a live-typing chat UI.

In [5]:
print('assistant: ', end='')
for chunk, meta in graph.stream(alert, stream_mode='messages'):
    print(chunk.content, end='', flush=True)   # print each token the instant it arrives
print(f'\n\n(tokens all came from node: {meta["langgraph_node"]!r})')

assistant: We spotted a HIGH-risk transaction on your account. We have paused it for review and sent a confirmation code to your phone.

(tokens all came from node: 'respond')


☝ The message printed **one token at a time**, not in a single lump — the same lazy, pull-one-at-a-time generator behaviour from the Python notebook, now sourced from inside a graph node. `meta['langgraph_node']` lets a multi-node graph label *which* node is speaking.

## 6. `.astream_events()` — the full firehose

`stream_mode` gives you **one** view. `astream_events()` gives you **all** of them at once: an async generator yielding a typed event for **every** lifecycle moment — chain start/end, node start/end, each model token, tool calls. You filter for the events you care about.

It's an **async** generator — so we drive it with `async for` (Day 03 + today's Python skill). Key event types:
- `on_chain_start` / `on_chain_end` — the graph or a node begins/finishes
- `on_chat_model_stream` — **one token** (the payload is in `event['data']['chunk']`)

In [6]:
async def show_events():
    async for ev in graph.astream_events(alert):
        kind = ev['event']
        if kind == 'on_chat_model_stream':
            tok = ev['data']['chunk'].content
            if tok:
                print(f'  token   : {tok!r}')
        elif kind in ('on_chain_start', 'on_chain_end'):
            print(f'{kind:16}: {ev["name"]}')

await show_events()

on_chain_start  : LangGraph
on_chain_start  : triage
on_chain_end    : triage
on_chain_start  : respond
  token   : 'We'
  token   : ' '
  token   : 'spotted'
  token   : ' '
  token   : 'a'
  token   : ' '
  token   : 'HIGH-risk'
  token   : ' '
  token   : 'transaction'
  token   : ' '
  token   : 'on'
  token   : ' '
  token   : 'your'
  token   : ' '
  token   : 'account.'
  token   : ' '
  token   : 'We'
  token   : ' '
  token   : 'have'
  token   : ' '
  token   : 'paused'
  token   : ' '
  token   : 'it'
  token   : ' '
  token   : 'for'
  token   : ' '
  token   : 'review'
  token   : ' '
  token   : 'and'
  token   : ' '
  token   : 'sent'
  token   : ' '
  token   : 'a'
  token   : ' '
  token   : 'confirmation'
  token   : ' '
  token   : 'code'
  token   : ' '
  token   : 'to'
  token   : ' '
  token   : 'your'
  token   : ' '
  token   : 'phone.'
on_chain_end    : respond
on_chain_end    : LangGraph


☝ One stream shows the **structure** (chain/node start→end) **and** the **content** (every token) interleaved in real time. `astream_events` is the debugging superpower: when an agent misbehaves you replay this stream to see exactly which node fired, what each model emitted, and where it stalled — the manual precursor to Day 08's LangSmith tracing.

## 7. A terminal UI that streams the agent's thoughts

Now assemble the payoff task: a small terminal renderer that shows **reasoning as it happens** — a node-level status line, then the answer typing out token by token. It consumes `astream_events` once and routes each event to the screen.

In [7]:
import sys

NODE_LABEL = {'triage': '\U0001f50d triaging alert', 'respond': '\u270d\ufe0f  drafting reply'}

async def stream_ui(alert_text: str):
    state = {'messages': [HumanMessage(content=alert_text)], 'risk': ''}
    print(f'\n\U0001f4e5 alert: {alert_text}\n' + '\u2500' * 48)
    async for ev in graph.astream_events(state):
        kind = ev['event']
        if kind == 'on_chain_start' and ev['name'] in NODE_LABEL:
            print(f'\n{NODE_LABEL[ev["name"]]} ...')
        elif kind == 'on_chat_model_stream':
            tok = ev['data']['chunk'].content
            sys.stdout.write(tok)              # type the answer out live
            sys.stdout.flush()
    print('\n' + '\u2500' * 48 + '\n\u2705 done')

await stream_ui('Overseas high-value purchase flagged')


📥 alert: Overseas high-value purchase flagged
────────────────────────────────────────────────

🔍 triaging alert ...

✍️  drafting reply ...
We spotted a HIGH-risk transaction on your account. We have paused it for review and sent a confirmation code to your phone.
────────────────────────────────────────────────
✅ done


☝ That's the whole UX win of streaming: the user sees **🔍 triaging** → **✍️ drafting** → the reply appearing word by word, instead of a blank screen then a sudden dump. Same graph as §3 — only the *consumer* of the stream changed. A real chat frontend does exactly this, pushing each token over a websocket.

## 8. Real LLM — stream actual Gemini tokens (needs `GOOGLE_API_KEY`)

Swap the fake model for `ChatGoogleGenerativeAI`. The graph, the nodes, and every stream call above are **unchanged** — that's the point: streaming is a property of the *graph runtime*, so a real model just drops in. (`streaming=True` makes Gemini emit token chunks instead of one final blob.)

In [8]:
if HAS_KEY:
    from langchain_google_genai import ChatGoogleGenerativeAI

    llm = ChatGoogleGenerativeAI(model='gemini-2.5-flash', temperature=0, streaming=True)

    def respond_real(state: TriageState) -> dict:
        prompt = (f'A {state["risk"]}-risk card transaction was flagged. '
                  'Write a short, calm 2-sentence message to the customer.')
        reply = llm.invoke([HumanMessage(content=prompt)])
        return {'messages': [reply]}

    b = StateGraph(TriageState)
    b.add_node('triage', triage)               # reuse the SAME triage node
    b.add_node('respond', respond_real)        # only the model changed
    b.add_edge(START, 'triage')
    b.add_edge('triage', 'respond')
    b.add_edge('respond', END)
    real_graph = b.compile()

    init = {'messages': [HumanMessage(content='Overseas high-value purchase flagged')], 'risk': ''}
    print('assistant: ', end='')
    for chunk, meta in real_graph.stream(init, stream_mode='messages'):
        print(chunk.content, end='', flush=True)
    print()
else:
    print('No GOOGLE_API_KEY \u2014 skipping. The fake-model sections above already show every streaming API.')

assistant: For your security, we've detected a high-risk transaction on your card. Please reply YES if authorized or NO if this was not you.


☝ Identical `stream_mode='messages'` loop as §5 — real Gemini tokens now flow through it. The lesson: **write the graph once, stream it any way you need.** The fake model let us learn the APIs keyless; production only changes the model line.

## 9. Recap

| Concept | What it is | Why it's used |
|---------|-----------|---------------|
| **`graph.stream` / `astream`** | run the graph as a (async) generator | see progress live instead of waiting for the final state |
| **`stream_mode='values'`** | yields the whole state per step | snapshot / debug full state as it evolves |
| **`stream_mode='updates'`** | yields each node's delta | node-by-node progress log |
| **`stream_mode='messages'`** | yields `(token, metadata)` | live token-by-token chat UI |
| **`.astream_events()`** | async generator of *every* event | full firehose: structure + tokens together, for UIs & debugging |
| **`on_chat_model_stream`** | the per-token event | pull each token from `event['data']['chunk']` |
| **`GenericFakeChatModel`** | a keyless model that streams | learn every streaming API with no API key |
| **`streaming=True`** (Gemini) | emit token chunks, not one blob | wire a real model into the same graph unchanged |

**The one idea:** a LangGraph run is a **generator of events** — the Python generator/async-generator skill from this morning *is* the streaming API.